# SignalUp AI — IoT Device User Upgrade Propensity Prediction

## Complete Machine Learning Notebook

**Project domain:** IoT & MarTech — AI-Driven Subscription Upgrade Propensity & Personalized Marketing  
**Goal:** Predict whether an IoT account/user is likely to upgrade to a premium subscription.

This notebook is designed for the synthetic SignalUp AI dataset:

```text
signalup_ai_dataset.csv
```

It includes:

- Dataset loading and validation
- Data quality checks
- Exploratory Data Analysis
- Feature engineering
- Data leakage handling
- Train/test split
- Preprocessing pipeline
- Multiple ML classification models
- XGBoost / LightGBM support when installed
- Model comparison
- Best model selection
- SHAP explainability support
- Customer-level predictions
- Marketing action segmentation
- Artifact export for backend integration

## 0. Important ML Notes

### Target column

```text
Actual_Upgrade_Occurred
```

This is the output label.

### Identifier column

```text
Account_ID
```

This column identifies the account but should not be used for training.

### Leakage-risk columns

The following columns should not be used as training inputs if they were generated using target-aware rules or previous models:

```text
Upgrade_Probability_Prior
```

It can be kept for reporting/comparison, but it is excluded from the training feature set by default.

In [ ]:
# Optional installation cell
# Run only if packages are missing in your environment.

# !pip install pandas numpy matplotlib scikit-learn joblib
# !pip install xgboost lightgbm shap

In [ ]:
import os
import json
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

import joblib

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

RANDOM_STATE = 42

In [ ]:
# Optional model libraries

optional_models_available = {}

try:
    from xgboost import XGBClassifier
    optional_models_available["xgboost"] = True
except Exception as exc:
    optional_models_available["xgboost"] = False
    print("XGBoost not available:", exc)

try:
    from lightgbm import LGBMClassifier
    optional_models_available["lightgbm"] = True
except Exception as exc:
    optional_models_available["lightgbm"] = False
    print("LightGBM not available:", exc)

try:
    import shap
    optional_models_available["shap"] = True
except Exception as exc:
    optional_models_available["shap"] = False
    print("SHAP not available:", exc)

optional_models_available

In [ ]:
# Project paths

DATA_PATH = "signalup_ai_dataset.csv"

ARTIFACT_DIR = Path("artifacts")
MODEL_DIR = ARTIFACT_DIR / "models"
PREPROCESSOR_DIR = ARTIFACT_DIR / "preprocessors"
METRICS_DIR = ARTIFACT_DIR / "metrics"
PREDICTION_DIR = ARTIFACT_DIR / "predictions"
EXPLAINABILITY_DIR = ARTIFACT_DIR / "explainability"

for path in [MODEL_DIR, PREPROCESSOR_DIR, METRICS_DIR, PREDICTION_DIR, EXPLAINABILITY_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Artifact folders ready.")

In [ ]:
# Load dataset

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}. Keep signalup_ai_dataset.csv in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

In [ ]:
# Basic schema validation

TARGET = "Actual_Upgrade_Occurred"
ID_COLS = ["Account_ID"]

required_columns = [
    "Account_ID",
    "Base_Hardware",
    "Connected_Device_Count",
    "Avg_Monthly_Data_Usage_GB",
    "Data_Tier_Ceiling_Hits_6M",
    "Upgrade_Page_Views_30D",
    "Marketing_Action_Segment",
    TARGET,
]

missing_required = [col for col in required_columns if col not in df.columns]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

print("Schema validation passed.")

# 1. Data Understanding and Quality Checks

In [ ]:
# Dataset overview

summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": df.isnull().sum().values,
    "missing_pct": (df.isnull().mean() * 100).round(2).values,
    "unique_count": df.nunique(dropna=True).values,
})

summary

In [ ]:
# Duplicate and target checks

print("Duplicate rows:", df.duplicated().sum())
print("\nTarget distribution:")
print(df[TARGET].value_counts(dropna=False))

print("\nTarget percentage:")
print((df[TARGET].value_counts(normalize=True) * 100).round(2))

In [ ]:
# Separate column types

categorical_cols_all = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols_all = df.select_dtypes(include=["int64", "float64", "int32", "float32", "bool"]).columns.tolist()

print("Categorical columns:", len(categorical_cols_all))
print(categorical_cols_all)

print("\nNumeric columns:", len(numeric_cols_all))
print(numeric_cols_all)

In [ ]:
# Numerical summary

df[numeric_cols_all].describe().T

In [ ]:
# Categorical summary

df[categorical_cols_all].describe().T

# 2. Exploratory Data Analysis

In [ ]:
# Target distribution chart

target_counts = df[TARGET].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(target_counts.index.astype(str), target_counts.values)
plt.title("Actual Upgrade Occurred Distribution")
plt.xlabel("Actual_Upgrade_Occurred")
plt.ylabel("Count")
plt.show()

In [ ]:
# Top categorical distributions

categorical_to_plot = [
    "Account_Type",
    "Company_Size",
    "Industry",
    "Region",
    "Current_Subscription",
    "Base_Hardware",
    "Marketing_Action_Segment",
    "Last_Campaign_Response",
]

for col in categorical_to_plot:
    if col in df.columns:
        counts = df[col].value_counts().head(10)
        plt.figure(figsize=(8, 4))
        plt.bar(counts.index.astype(str), counts.values)
        plt.title(f"Distribution of {col}")
        plt.xticks(rotation=45, ha="right")
        plt.ylabel("Count")
        plt.tight_layout()
        plt.show()

In [ ]:
# Numeric distributions for important variables

numeric_to_plot = [
    "Connected_Device_Count",
    "Avg_Monthly_Data_Usage_GB",
    "Data_Tier_Ceiling_Hits_6M",
    "Upgrade_Page_Views_30D",
    "Customer_Engagement_Score",
    "Device_Usage_Score",
    "Premium_Interest_Score",
    "Customer_Health_Score",
]

for col in numeric_to_plot:
    if col in df.columns:
        plt.figure(figsize=(7, 4))
        plt.hist(df[col].dropna(), bins=40)
        plt.title(f"Distribution of {col}")
        plt.xlabel(col)
        plt.ylabel("Frequency")
        plt.tight_layout()
        plt.show()

In [ ]:
# Upgrade rate by selected categorical columns

for col in categorical_to_plot:
    if col in df.columns:
        upgrade_rate = df.groupby(col)[TARGET].mean().sort_values(ascending=False).head(15)
        plt.figure(figsize=(8, 4))
        plt.bar(upgrade_rate.index.astype(str), upgrade_rate.values)
        plt.title(f"Upgrade Rate by {col}")
        plt.xticks(rotation=45, ha="right")
        plt.ylabel("Upgrade Rate")
        plt.tight_layout()
        plt.show()

In [ ]:
# Correlation with target for numeric features

numeric_features_for_corr = [col for col in numeric_cols_all if col != TARGET]

corr_with_target = (
    df[numeric_features_for_corr + [TARGET]]
    .corr(numeric_only=True)[TARGET]
    .drop(TARGET)
    .sort_values(ascending=False)
)

corr_with_target.head(20)

# 3. Feature Selection and Leakage Control

In [ ]:
# Columns excluded from training

LEAKAGE_COLS = [
    "Upgrade_Probability_Prior",   # prior score generated from synthetic rule; do not train on it
    "Propensity_Score_Calculated", # old column name if present
]

DROP_COLS = ID_COLS + [TARGET] + [col for col in LEAKAGE_COLS if col in df.columns]

feature_cols = [col for col in df.columns if col not in DROP_COLS]

X = df[feature_cols].copy()
y = df[TARGET].astype(int).copy()

categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

print("Total features used:", len(feature_cols))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Dropped columns:", DROP_COLS)

In [ ]:
# Train/test split

X_train, X_test, y_train, y_test, train_index, test_index = train_test_split(
    X,
    y,
    df.index,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

# 4. Preprocessing Pipeline

In [ ]:
# Preprocessing pipelines

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

print("Preprocessor ready.")

# 5. Model Training

In [ ]:
# Define models

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "DecisionTree": DecisionTreeClassifier(max_depth=8, class_weight="balanced", random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(
        n_estimators=150,
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=150,
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
}

if optional_models_available.get("xgboost"):
    models["XGBoost"] = XGBClassifier(
        n_estimators=250,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

if optional_models_available.get("lightgbm"):
    models["LightGBM"] = LGBMClassifier(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

list(models.keys())

In [ ]:
def evaluate_classifier(name, pipeline, X_test, y_test):
    y_pred = pipeline.predict(X_test)

    if hasattr(pipeline, "predict_proba"):
        y_proba = pipeline.predict_proba(X_test)[:, 1]
    else:
        y_proba = y_pred

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
    }

    return metrics, y_pred, y_proba

In [ ]:
# Train and evaluate models

results = []
trained_pipelines = {}
predictions_cache = {}

for name, model in models.items():
    print(f"\nTraining {name}...")

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    pipeline.fit(X_train, y_train)

    metrics, y_pred, y_proba = evaluate_classifier(name, pipeline, X_test, y_test)

    results.append(metrics)
    trained_pipelines[name] = pipeline
    predictions_cache[name] = {
        "y_pred": y_pred,
        "y_proba": y_proba,
    }

    print(metrics)

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
results_df

In [ ]:
# Select best model

best_model_name = results_df.iloc[0]["model"]
best_pipeline = trained_pipelines[best_model_name]

print("Best model:", best_model_name)
results_df.iloc[0].to_dict()

# 6. Detailed Evaluation of Best Model

In [ ]:
best_pred = predictions_cache[best_model_name]["y_pred"]
best_proba = predictions_cache[best_model_name]["y_proba"]

print("Classification Report:")
print(classification_report(y_test, best_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, best_pred))

In [ ]:
# ROC curve

fpr, tpr, _ = roc_curve(y_test, best_proba)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"{best_model_name}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Precision-Recall curve

precision, recall, _ = precision_recall_curve(y_test, best_proba)

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, label=f"{best_model_name}")
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.grid(True)
plt.show()

# 7. Marketing Segmentation from Prediction Scores

In [ ]:
def propensity_category(score):
    if score >= 0.80:
        return "VERY_HIGH"
    if score >= 0.60:
        return "HIGH"
    if score >= 0.40:
        return "MEDIUM"
    if score >= 0.20:
        return "LOW"
    return "VERY_LOW"


def recommended_marketing_action(category):
    mapping = {
        "VERY_HIGH": "Immediate Premium Trial Offer",
        "HIGH": "Discount + Premium Feature Campaign",
        "MEDIUM": "Educational Nurture Campaign",
        "LOW": "Re-engagement Campaign",
        "VERY_LOW": "Wait and Monitor",
    }
    return mapping.get(category, "Wait and Monitor")


test_output = df.loc[test_index, ["Account_ID"]].copy()
test_output["actual_upgrade"] = y_test.values
test_output["predicted_upgrade"] = best_pred
test_output["upgrade_probability"] = best_proba
test_output["propensity_category"] = test_output["upgrade_probability"].apply(propensity_category)
test_output["recommended_marketing_action"] = test_output["propensity_category"].apply(recommended_marketing_action)

test_output.head()

In [ ]:
# Propensity category distribution

category_counts = test_output["propensity_category"].value_counts()

plt.figure(figsize=(7, 4))
plt.bar(category_counts.index.astype(str), category_counts.values)
plt.title("Predicted Propensity Category Distribution")
plt.xlabel("Propensity Category")
plt.ylabel("Count")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

category_counts

# 8. Feature Importance

In [ ]:
def get_feature_names_from_preprocessor(preprocessor):
    output_features = []

    if numeric_features:
        output_features.extend(numeric_features)

    if categorical_features:
        onehot = preprocessor.named_transformers_["cat"].named_steps["onehot"]
        cat_names = onehot.get_feature_names_out(categorical_features).tolist()
        output_features.extend(cat_names)

    return output_features


feature_names = get_feature_names_from_preprocessor(best_pipeline.named_steps["preprocessor"])

model_step = best_pipeline.named_steps["model"]

importance_df = None

if hasattr(model_step, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": model_step.feature_importances_,
    }).sort_values("importance", ascending=False)

elif hasattr(model_step, "coef_"):
    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": np.abs(model_step.coef_[0]),
    }).sort_values("importance", ascending=False)

if importance_df is not None:
    display(importance_df.head(30))

    top_imp = importance_df.head(20).sort_values("importance")
    plt.figure(figsize=(8, 6))
    plt.barh(top_imp["feature"], top_imp["importance"])
    plt.title(f"Top Feature Importance — {best_model_name}")
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance is not available for this model.")

# 9. SHAP Explainability

This section runs only if SHAP is installed. SHAP may take time on large datasets, so we use a sample.

In [ ]:
shap_summary_path = EXPLAINABILITY_DIR / "shap_summary.csv"

if optional_models_available.get("shap") and best_model_name in ["RandomForest", "ExtraTrees", "XGBoost", "LightGBM", "DecisionTree", "GradientBoosting"]:
    try:
        X_train_transformed = best_pipeline.named_steps["preprocessor"].transform(X_train.sample(min(1000, len(X_train)), random_state=RANDOM_STATE))
        X_test_sample = X_test.sample(min(500, len(X_test)), random_state=RANDOM_STATE)
        X_test_transformed = best_pipeline.named_steps["preprocessor"].transform(X_test_sample)

        explainer = shap.Explainer(best_pipeline.named_steps["model"], X_train_transformed)
        shap_values = explainer(X_test_transformed)

        shap_abs = np.abs(shap_values.values).mean(axis=0)

        if shap_abs.ndim > 1:
            shap_abs = shap_abs[:, -1]

        shap_df = pd.DataFrame({
            "feature": feature_names,
            "mean_abs_shap": shap_abs,
        }).sort_values("mean_abs_shap", ascending=False)

        shap_df.to_csv(shap_summary_path, index=False)

        display(shap_df.head(20))

        top_shap = shap_df.head(20).sort_values("mean_abs_shap")
        plt.figure(figsize=(8, 6))
        plt.barh(top_shap["feature"], top_shap["mean_abs_shap"])
        plt.title("Top SHAP Feature Contributions")
        plt.tight_layout()
        plt.show()

    except Exception as exc:
        print("SHAP execution skipped due to error:", exc)
else:
    print("SHAP not installed or not suitable for selected model. Skipping SHAP section.")

# 10. Save Artifacts

In [ ]:
# Save model pipeline and metadata

model_path = MODEL_DIR / "signalup_upgrade_propensity_model.joblib"
metadata_path = METRICS_DIR / "model_metadata.json"
metrics_path = METRICS_DIR / "model_metrics.json"
predictions_path = PREDICTION_DIR / "test_predictions.csv"
feature_importance_path = EXPLAINABILITY_DIR / "feature_importance.csv"

joblib.dump(best_pipeline, model_path)

model_metrics = results_df.to_dict(orient="records")

metadata = {
    "project": "SignalUp AI",
    "target": TARGET,
    "best_model": best_model_name,
    "model_path": str(model_path),
    "trained_at": datetime.utcnow().isoformat() + "Z",
    "dataset_rows": int(len(df)),
    "dataset_columns": int(len(df.columns)),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "features_used": feature_cols,
    "dropped_columns": DROP_COLS,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "propensity_categories": {
        "VERY_HIGH": "score >= 0.80",
        "HIGH": "0.60 <= score < 0.80",
        "MEDIUM": "0.40 <= score < 0.60",
        "LOW": "0.20 <= score < 0.40",
        "VERY_LOW": "score < 0.20",
    },
}

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

with open(metrics_path, "w") as f:
    json.dump(model_metrics, f, indent=4)

test_output.to_csv(predictions_path, index=False)

if importance_df is not None:
    importance_df.to_csv(feature_importance_path, index=False)

print("Artifacts saved:")
print("Model:", model_path)
print("Metadata:", metadata_path)
print("Metrics:", metrics_path)
print("Predictions:", predictions_path)
print("Feature importance:", feature_importance_path if importance_df is not None else "Not available")

# 11. Single Customer Prediction Example

In [ ]:
def predict_single_account(account_id: str):
    row = df[df["Account_ID"] == account_id].copy()

    if row.empty:
        raise ValueError(f"Account not found: {account_id}")

    X_single = row[feature_cols]
    probability = best_pipeline.predict_proba(X_single)[:, 1][0]
    prediction = int(probability >= 0.5)
    category = propensity_category(probability)
    action = recommended_marketing_action(category)

    return {
        "Account_ID": account_id,
        "prediction": prediction,
        "upgrade_probability": round(float(probability), 4),
        "propensity_category": category,
        "recommended_marketing_action": action,
    }


sample_account = df["Account_ID"].iloc[0]
predict_single_account(sample_account)

# 12. Backend Integration Notes

The saved model can be loaded inside your FastAPI prediction service using:

```python
import joblib
pipeline = joblib.load("artifacts/models/signalup_upgrade_propensity_model.joblib")
probability = pipeline.predict_proba(input_df)[:, 1][0]
```

Recommended backend output:

```json
{
  "customer_id": "CUS-XXXX",
  "prediction_score": 0.84,
  "propensity_category": "VERY_HIGH",
  "recommended_action": "Immediate Premium Trial Offer"
}
```